# Tooth-AI paper evaluation

Runs Tasks D and E on the 716-image stratified test split.

**Run the cells in order.** Cell 1 must show a GPU or everything below is
meaningless: the Kaggle push API has been reported to hand out a different
machine than the one requested without raising, so this notebook checks rather
than trusts.


## 1. GPU gate — fails loudly if there is no GPU


In [ ]:
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

try:
    name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
except Exception as exc:
    raise SystemExit(f"FATAL: no nvidia-smi -- this session has no GPU ({exc})")

if not name:
    raise SystemExit("FATAL: nvidia-smi returned no GPU")

print("GPU:", name)
GPU_NAME = name.split(",")[0].strip()
print("\nThis exact string goes into every results CSV as the latency hardware.")


## 2. Install the pinned stack

The checkpoints were trained and used with **torch 2.5.1+cu121** and
**Detectron2 0.6 at commit `fd27788`** (read from the project's own
`dist-info`/`direct_url.json`). We try those first.

Any deviation prints a loud warning — small numeric differences follow from a
different torch build, and the paper must not silently report numbers from an
unpinned stack.

This cell is deliberately alone so it can be re-run and debugged by itself.


In [ ]:
import subprocess, sys, warnings

PIN_TORCH        = "2.5.1"
PIN_TV           = "0.20.1"
PIN_CUDA         = "cu121"
PIN_D2_COMMIT    = "fd27788985af0f4ca800bca563acdb700bb890e2"

def sh(cmd):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, text=True).returncode

DEVIATIONS = []

# --- torch -----------------------------------------------------------------
rc = sh([sys.executable, "-m", "pip", "install", "-q",
         f"torch=={PIN_TORCH}+{PIN_CUDA}", f"torchvision=={PIN_TV}+{PIN_CUDA}",
         "--index-url", f"https://download.pytorch.org/whl/{PIN_CUDA}"])
if rc != 0:
    DEVIATIONS.append(f"torch {PIN_TORCH}+{PIN_CUDA} would not install; kept the preinstalled build")
    print("!! pinned torch failed to install -- continuing with whatever is present")

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("FATAL: torch cannot see the GPU")
if not torch.__version__.startswith(PIN_TORCH):
    DEVIATIONS.append(f"torch is {torch.__version__}, pinned {PIN_TORCH}+{PIN_CUDA}")


### 2b. Detectron2 at the pinned commit


In [ ]:
# Build the exact commit the checkpoints were produced with. If that build
# fails (CUDA/toolchain drift on the Kaggle image), fall back to main and say so.
rc = sh([sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/facebookresearch/detectron2.git@{PIN_D2_COMMIT}"])
if rc != 0:
    DEVIATIONS.append(f"Detectron2 {PIN_D2_COMMIT[:7]} would not build; used main instead")
    print("!! pinned Detectron2 commit failed to build -- falling back to main")
    rc = sh([sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/facebookresearch/detectron2.git"])
    if rc != 0:
        raise SystemExit("FATAL: Detectron2 would not build at all")

sh([sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "pycocotools"])


### 2c. Verify — and shout about any deviation


In [ ]:
import torch, detectron2, cv2, numpy, pycocotools
print("torch      ", torch.__version__)
print("detectron2 ", detectron2.__version__)
print("opencv     ", cv2.__version__)
print("numpy      ", numpy.__version__)
print("GPU        ", torch.cuda.get_device_name(0))

from detectron2.engine import DefaultPredictor   # import must actually work
print("\ndetectron2 imports cleanly")

if DEVIATIONS:
    print("\n" + "!" * 72)
    print("VERSION DEVIATION FROM THE PINNED STACK")
    for d in DEVIATIONS:
        print("  -", d)
    print("Numbers from this run may differ slightly from the pinned stack.")
    print("Record this in the paper's reproducibility note.")
    print("!" * 72)
else:
    print("\nStack matches the pins exactly.")


## 3. Locate the dataset

Attach the private Kaggle Dataset built by `make_bundle.py`. It appears under
`/kaggle/input/<slug>/`.


In [ ]:
import os, glob, shutil, sys

# Kaggle expands uploaded .zip members, so the bundle's top-level directories
# (data/, output/, paper_eval/) appear directly under /kaggle/input/<slug>/.
cands = sorted(glob.glob("/kaggle/input/*/paper_eval"))
if not cands:
    print("Contents of /kaggle/input:")
    for d in sorted(glob.glob("/kaggle/input/*")):
        print("  ", d, sorted(os.listdir(d))[:12])
    raise SystemExit("FATAL: bundle not attached, or paper_eval/ not found in it. "
                     "Add the private dataset to this notebook.")
BUNDLE = os.path.dirname(cands[0])
print("bundle:", BUNDLE)
for need in ("paper_eval", "output", "data/test"):
    ok = os.path.isdir(os.path.join(BUNDLE, need))
    print(f"  {need:<12} {'OK' if ok else 'MISSING'}")
    if not ok:
        raise SystemExit(f"FATAL: {need} missing from the bundle")

WORK = "/kaggle/working/tooth"
os.makedirs(WORK, exist_ok=True)
# paper_eval writes results, so copy the code out of the read-only input mount
dst = os.path.join(WORK, "paper_eval")
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(os.path.join(BUNDLE, "paper_eval"), dst)

# the harness expects data/final-di-stratified/test and output/<model>/
os.makedirs(os.path.join(WORK, "data/final-di-stratified"), exist_ok=True)
link = os.path.join(WORK, "data/final-di-stratified/test")
if not os.path.exists(link):
    os.symlink(os.path.join(BUNDLE, "data/test"), link)
outlink = os.path.join(WORK, "output")
if not os.path.exists(outlink):
    os.symlink(os.path.join(BUNDLE, "output"), outlink)

os.chdir(WORK)
print("\ncwd:", os.getcwd())

# The repo lives on an exFAT volume, which sprays AppleDouble "._<name>"
# sidecars beside every file. Those rode along into the dataset and they match
# a *.jpg glob, so count real images explicitly.
jpgs = [f for f in glob.glob("data/final-di-stratified/test/*.jpg")
        if not os.path.basename(f).startswith("._")]
print("test images:", len(jpgs), "(expected 716)")
assert len(jpgs) == 716, f"expected 716 test images, found {len(jpgs)}"

import json as _json
ann = "data/final-di-stratified/test/_annotations.coco.json"
meta = _json.load(open(ann))
print("annotations:", len(meta["images"]), "images,", len(meta["annotations"]), "anns")

for m in ("resnet50_9class_20k", "rtx4060_48k", "resnext101_cascade_60k"):
    w = f"output/{m}/model_final.pth"
    print(f"  {m:<26} {'OK' if os.path.exists(w) else 'MISSING'}  "
          f"{os.path.getsize(w)/1e6:.0f} MB" if os.path.exists(w) else f"  {m}: MISSING")


## 4. Smoke test — 5 images

Proves the whole harness runs before spending an hour on the full split.
Check the printed table is populated, then move on.


In [ ]:
# RUN_FULL gates every long cell below. Left False for the first push so the
# session proves the harness works on 5 images before an hour of GPU time is
# spent. Flip to True and re-push (or re-run interactively) for the real run.
RUN_FULL = False

!python3 paper_eval/run_eval.py --all --smoke-test --latency


## 5. Full run — all configurations


In [ ]:
if not RUN_FULL:
    print("SKIPPED (full ablation) -- RUN_FULL is False. Set it in the smoke-test cell.")
else:
    !python3 paper_eval/run_eval.py --all --latency


## 6. Threshold sweep


In [ ]:
if not RUN_FULL:
    print("SKIPPED (threshold sweep) -- RUN_FULL is False. Set it in the smoke-test cell.")
else:
    !python3 paper_eval/run_eval.py --sweep


## 7. COCO re-evaluation on the 716 split

Three points: R50 9-class, R50 41-class, X101 Cascade 41-class — each at its
own configured test size, which the CSV records.


In [ ]:
if not RUN_FULL:
    print("SKIPPED (COCO re-evaluation) -- RUN_FULL is False. Set it in the smoke-test cell.")
else:
    !python3 paper_eval/run_eval.py --coco-eval


## 8. Figure candidates and panels


In [ ]:
if not RUN_FULL:
    print("SKIPPED (panel candidates) -- RUN_FULL is False. Set it in the smoke-test cell.")
else:
    !python3 paper_eval/export_panels.py --suggest-images


In [ ]:
# Pick an image_id from the table above -- ideally one flagged as introducing
# a central-zone false positive -- and export its panels.
IMAGE_ID = None          # <-- set me

if IMAGE_ID is None:
    print("set IMAGE_ID from the candidate table above, then re-run this cell")
else:
    !python3 paper_eval/export_panels.py --image-id {IMAGE_ID}


## 9. Zip the results


In [ ]:
import shutil, os
out = "/kaggle/working/paper_eval_results"
shutil.make_archive(out, "zip", "paper_eval/results")
print(out + ".zip", os.path.getsize(out + ".zip"), "bytes")
for root, _d, files in os.walk("paper_eval/results"):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.getsize(p):>10}  {p}")
